In [1]:
import os
import shutil
import glob

%cd /kaggle/working

if os.path.exists("benchmarking-generative-models-for-domain-adaptation"):
    shutil.rmtree("benchmarking-generative-models-for-domain-adaptation")

!git clone --recursive https://github.com/AntonioRosano/benchmarking-generative-models-for-domain-adaptation.git

repo_path = "/kaggle/working/benchmarking-generative-models-for-domain-adaptation/models/unit"
%cd {repo_path}

!pip install -q tensorboardX

/kaggle/working
Cloning into 'benchmarking-generative-models-for-domain-adaptation'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 119 (delta 49), reused 82 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 3.28 MiB | 14.49 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Submodule 'models/cut' (https://github.com/taesungp/contrastive-unpaired-translation) registered for path 'models/cut'
Submodule 'models/munit' (https://github.com/NVlabs/MUNIT) registered for path 'models/munit'
Submodule 'models/pspnet' (https://github.com/hszhao/semseg) registered for path 'models/pspnet'
Submodule 'models/unit' (https://github.com/mingyuliutw/UNIT) registered for path 'models/unit'
Cloning into '/kaggle/working/benchmarking-generative-models-for-domain-adaptation/models/cut'...
remote: Enumerating objects: 286, done.        
remote: Counting objects: 100% (144/144), done

In [ ]:
UTILS_PATH = "utils.py"

if os.path.exists(UTILS_PATH):
    with open(UTILS_PATH, "r") as f:
        lines = f.readlines()
    
    with open(UTILS_PATH, "w") as f:
        for line in lines:
            if "from torch.utils.serialization import load_lua" in line:
                f.write("# " + line)
            
            elif "return yaml.load(stream)" in line:
                f.write(line.replace("yaml.load(stream)", "yaml.load(stream, Loader=yaml.FullLoader)"))
            
            elif "yaml.safe_load(stream)" in line:
                f.write(line.replace("yaml.safe_load(stream)", "yaml.load(stream, Loader=yaml.FullLoader)"))
            
            else:
                f.write(line)
                
    print("utils.py corretto: rimosso load_lua e aggiornato YAML Loader.")
else:
    print("File utils.py non trovato!")

✅ utils.py corretto: rimosso load_lua e aggiornato YAML Loader.


In [3]:
BASE_IN = "/kaggle/input"
REAL_SOURCE = SYN_SOURCE = ""

for r, d, f in os.walk(BASE_IN):
    if r.endswith("real/train/frames"): REAL_SOURCE = r
    if r.endswith("synthetic/train/frames"): SYN_SOURCE = r

DEST_DATASET = os.path.join(os.getcwd(), "datasets", "ego_adaptation")
dirs = {k: os.path.join(DEST_DATASET, k) for k in ["trainA", "trainB", "testA", "testB"]}

if os.path.exists(DEST_DATASET): shutil.rmtree(DEST_DATASET)
for d in dirs.values(): os.makedirs(d, exist_ok=True)

def copy_all(src, dst):
    if not src or not os.path.exists(src): return 0
    files = [f for f in glob.glob(os.path.join(src, "*.*")) if os.path.isfile(f)]
    for f in files: shutil.copy(f, dst)
    return len(files)

print(f"Copia REALI: {copy_all(REAL_SOURCE, dirs['trainA'])}")
print(f"Copia SINTETICHE: {copy_all(SYN_SOURCE, dirs['trainB'])}")

def pop_test(src, dst, c=10):
    for f in glob.glob(os.path.join(src, "*.*"))[:c]: shutil.copy(f, dst)

pop_test(dirs['trainA'], dirs['testA'])
pop_test(dirs['trainB'], dirs['testB'])

print(f"\nDATASET PRONTO.")
for k, v in dirs.items(): print(f"{k}: {len(os.listdir(v))}")

Copia REALI: 4740
Copia SINTETICHE: 12000

DATASET PRONTO.
trainA: 4740
trainB: 12000
testA: 10
testB: 10


In [ ]:
import os

config_file = "configs/unit_gta2city_folder.yaml"
DEST_DATASET = os.path.join(os.getcwd(), "datasets", "ego_adaptation")

unit_config = f"""
# UNIT Config - Modalità Ultra-Realismo
image_save_iter: 1000
image_display_iter: 500
display_size: 4
snapshot_save_iter: 5000
log_iter: 100

max_iter: 1000000
batch_size: 4
weight_decay: 0.0001
beta1: 0.5
beta2: 0.999
init: kaiming
lr: 0.0001
lr_policy: step
step_size: 100000
gamma: 0.5

# Pesi Bilanciati per Texture Nitide
gan_w: 10            # Raddoppiato: la pressione del realismo è massima
recon_x_w: 5         # Ridotto: meno vincolo "pittorico"
recon_h_w: 10
recon_kl_w: 0.01
recon_x_cyc_w: 5     
recon_kl_cyc_w: 0.01
vgg_w: 1             # ATTIVATO: Percepisce bordi e texture reali

gen:
  dim: 64
  activ: lrelu       # Cambiato in Leaky ReLU per evitare stallo
  n_downsample: 2
  n_res: 4           # Se hai VRAM, potresti portarlo a 6 per più "cervello"
  pad_type: reflect
dis:
  dim: 64
  norm: none
  activ: lrelu
  n_layer: 5         # Discriminatore più profondo = più severo
  gan_type: lsgan
  num_scales: 3      
  pad_type: reflect

input_dim_a: 3
input_dim_b: 3
num_workers: 2
new_size: 256                # Ridimensiona il lato corto
crop_image_height: 144       # Altezza (16:9 rispetto a 256)
crop_image_width: 256

data_root: {DEST_DATASET}
"""

with open(config_file, "w") as f:
    f.write(unit_config)
print(f"Config 'Realism-Boost' (16:9) salvato in: {config_file}")

✅ Config 'Realism-Boost' (16:9) salvato in: configs/unit_gta2city_folder.yaml


In [ ]:
checkpoint_dir = "./outputs/ego_unit_kaggle/outputs/unit_gta2city_folder/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
 
src_dir = "/kaggle/input/datasets/angelospadola/iiiiiiii/a"
 
for f in os.listdir(src_dir):
    if f.endswith(".pt"):
        shutil.copy(os.path.join(src_dir, f), os.path.join(checkpoint_dir, f))
        print(f"Copiato {f} in {checkpoint_dir}")
 
TRAINER_PATH = "trainer.py"
with open(TRAINER_PATH, "r") as f:
    t_content = f.read()

t_content = t_content.replace("self.dis_opt.load_state_dict(state_dict['dis'])", 
                              "if 'dis' in state_dict: self.dis_opt.load_state_dict(state_dict['dis'])")
t_content = t_content.replace("self.gen_opt.load_state_dict(state_dict['gen'])", 
                              "if 'gen' in state_dict: self.gen_opt.load_state_dict(state_dict['gen'])")

with open(TRAINER_PATH, "w") as f:
    f.write(t_content)
print("trainer.py patchato per un resume sicuro.")

✅ Copiato dis_00045000.pt in ./outputs/ego_unit_kaggle/outputs/unit_gta2city_folder/checkpoints
✅ Copiato gen_00045000.pt in ./outputs/ego_unit_kaggle/outputs/unit_gta2city_folder/checkpoints
✅ Copiato optimizer.pt in ./outputs/ego_unit_kaggle/outputs/unit_gta2city_folder/checkpoints
✅ trainer.py patchato per un resume sicuro.


In [6]:
!pip install tensorboardX
!python train.py \
  --trainer UNIT \
  --resume \
  --config configs/unit_gta2city_folder.yaml \
  --output_path ./outputs/ego_unit_kaggle

--2026-03-01 22:12:21--  https://www.dropbox.com/s/76l3rt4kyi3s8x7/vgg16.t7?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.3.18, 2620:100:6018:18::a27d:312
Connecting to www.dropbox.com (www.dropbox.com)|162.125.3.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/g9wy1a9dsxvaggiausowr/vgg16.t7?rlkey=ru9atqe9inqquft2dgesl2kvj&dl=1 [following]
--2026-03-01 22:12:21--  https://www.dropbox.com/scl/fi/g9wy1a9dsxvaggiausowr/vgg16.t7?rlkey=ru9atqe9inqquft2dgesl2kvj&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uce45981e0b83576a565e9901efd.dl.dropboxusercontent.com/cd/0/inline/C71asLhe-E51o3l0FAUNT9Ga9bTdN9i9LePoccad5PRoEcwXAc3lxlldIirha0y2UyirvRoTk9yJINN6f6LmlBTqgyqzLhZtOSK6-wxAW6TqyUyVWSKUmv8AkdxmmjJdYJQ/file?dl=1# [following]
--2026-03-01 22:12:22--  https://uce45981e0b83576a565e9901efd.dl.dropboxusercontent.com/cd/0/inline/C71asLhe-E51o3